In [1]:
import pandas as pd

# Load the raw telemetry matrix
adsb_df = pd.read_csv("live_raw_adsb2.csv")

print("--- ADS-B Dataset Profile ---")
print(f"Total rows captured: {adsb_df.shape[0]}")
print(f"Total columns: {adsb_df.shape[1]}\n")

print("Missing values per attribute:")
print(adsb_df.isnull().sum())

# Display descriptive statistics of aircraft dynamics
print("\nTelemetry Distribution Summary:")
adsb_df[['latitude', 'longitude', 'baro_altitude', 'velocity']].describe()

--- ADS-B Dataset Profile ---
Total rows captured: 760138
Total columns: 8

Missing values per attribute:
timestamp             0
icao24                0
callsign          11453
origin_country     5025
longitude          7484
latitude           7484
baro_altitude     63247
velocity            249
dtype: int64

Telemetry Distribution Summary:


,latitude,longitude,baro_altitude,velocity
count,752654.000000,752654.000000,696891.000000,759889.000000
mean,38.152308,-53.096295,6487.916368,152.882183
std,12.070196,61.124429,4508.996279,88.778433
min,-52.635300,-173.049000,-304.800000,0.000000
25%,33.862700,-97.220900,1584.960000,62.640000
50%,39.289600,-80.392050,7482.840000,185.350000
75%,44.477300,3.398675,10675.620000,231.400000
max,78.889500,179.691100,38343.840000,1295.340000


In [3]:
# Load the raw textual communication messages
acars_df = pd.read_json("live_raw_acars2.json", lines=True)

print("--- ACARS Dataset Profile ---")
print(f"Total messages captured: {acars_df.shape[0]}")
print(f"Available attributes: {list(acars_df.columns)}\n")

# Peek at the actual text payloads sent by pilots during your run
print("Sample Message Payloads:")
print(acars_df[['fromHex', 'text']].head(10))

--- ACARS Dataset Profile ---
Total messages captured: 1069
Available attributes: ['id', 'station', 'airframe', 'flight', 'source', 'sourceType', 'linkDirection', 'fromHex', 'toHex', 'timestamp', 'tail', 'flightNumber', 'channel', 'frequency', 'level', 'error', 'mode', 'label', 'blockId', 'messageNumber', 'ack', 'data', 'text', 'departingAirport', 'destinationAirport', 'latitude', 'longitude', 'altitude', 'uuid', 'arUUID', 'createdAt', 'updatedAt']

Sample Message Payloads:
  fromHex                                               text
0    None  0 87.22 87.44 91.75 91.56 -53.\r\n3  390370.75...
1    None  MIL  \r\nR N1 VIBES                 0.2 MIL  \...
2    None                      201724 LFPG KATL7\r\n/FN 0085
3    None  0 87.22 87.44 91.75 91.56 -53.\r\n3  390370.75...
4    None  0 87.22 87.44 91.75 91.56 -53.\r\n3  390370.75...
5    None  1,0,1/\r\nS11,0,1/\r\n1S1,0,1/\r\nS21,0,1/\r\n...
6    None  MIL  \r\nR N1 VIBES                 0.2 MIL  \...
7    None  7.\r\n510.390 12672 12

In [4]:
# 1. Standardize joining keys to clean lowercase strings
adsb_df['icao24'] = adsb_df['icao24'].str.strip().str.lower()
acars_df['fromHex'] = acars_df['fromHex'].str.strip().str.lower()

# 2. Extract nested callsigns if present in the raw ACARS JSON object
if 'flight' in acars_df.columns:
    acars_df['acars_callsign'] = acars_df['flight'].apply(
        lambda x: x.get('callsign', '').strip() if isinstance(x, dict) else ''
    )

print("[+] Structural normalization complete. Primary keys are fully aligned!")

[+] Structural normalization complete. Primary keys are fully aligned!


In [6]:
# ==========================================
# STEP 3b: RESCUING MISSING IDENTITY KEYS
# ==========================================

print(f"Original missing keys in fromHex: {acars_df['fromHex'].isnull().sum()} out of {len(acars_df)}")

# If fromHex is missing, look inside the 'airframe' or 'station' objects for an alternative map
def rescue_hex(row):
    if pd.notnull(row['fromHex']) and str(row['fromHex']).lower() != 'none':
        return str(row['fromHex']).strip().lower()

    # Fallback to alternative keys if nested objects exist
    for alternate_key in ['airframe', 'station', 'tail']:
        val = row.get(alternate_key)
        if isinstance(val, dict) and 'hex' in val:
            return str(val['hex']).strip().lower()
        elif isinstance(val, str) and len(val) == 6: # Standard Hex length
            return val.strip().lower()

    return None

# Apply the fallback function
acars_df['icao24'] = acars_df.apply(rescue_hex, axis=1)

print(f"Remaining missing tracking keys after rescue: {acars_df['icao24'].isnull().sum()}")

Original missing keys in fromHex: 517 out of 1069
Remaining missing tracking keys after rescue: 68


In [8]:
# ==========================================
# SECTION 4: THE CROSS-STREAM DATA FUSION (FIXED)
# ==========================================

# 1. Perform an inner join on the matching unique aircraft hardware ID
fused_df = pd.merge(
    adsb_df,
    acars_df,
    on='icao24',
    suffixes=('_adsb', '_acars')
)

# --- DATATYPE NORMALIZATION STEP ---
# Force both timestamp series to be raw numerical floats/integers representing Unix epoch seconds
fused_df['timestamp_adsb_numeric'] = pd.to_numeric(fused_df['timestamp_adsb'])

# Convert the Datetime values into Unix Epoch seconds by casting to integer timestamps
if pd.api.types.is_datetime64_any_dtype(fused_df['timestamp_acars']):
    fused_df['timestamp_acars_numeric'] = fused_df['timestamp_acars'].astype('int64') // 10**9
else:
    fused_df['timestamp_acars_numeric'] = pd.to_numeric(fused_df['timestamp_acars'])
# ------------------------------------

# 2. Calculate the absolute time difference using our matching numerical formats
fused_df['time_delta'] = (fused_df['timestamp_adsb_numeric'] - fused_df['timestamp_acars_numeric']).abs()

# 3. Apply the strict 60-second temporal constraint filter
final_verified_stream = fused_df[fused_df['time_delta'] <= 60].reset_index(drop=True)

print("--- Data Fusion Complete ---")
print(f"Total synchronized cross-channel records captured: {final_verified_stream.shape[0]}\n")

# Display a sneak peek of your unified flight tracking sheet
if final_verified_stream.shape[0] > 0:
    print("Sample Synced Flight Log Profiles:")
    display_cols = ['timestamp_adsb', 'icao24', 'callsign', 'latitude', 'longitude', 'baro_altitude', 'text', 'time_delta']
    existing_cols = [c for c in display_cols if c in final_verified_stream.columns]
    print(final_verified_stream[existing_cols].head(5))
else:
    print("[!] No overlapping aircraft matches found within the 60-second window.")

--- Data Fusion Complete ---
Total synchronized cross-channel records captured: 1406

Sample Synced Flight Log Profiles:
   timestamp_adsb  icao24  callsign  baro_altitude  \
0      1781975663  440062  EJU68WB         9738.36   
1      1781975663  899043  CAL057         11277.60   
2      1781975663  8991e5  SJX031         12192.00   
3      1781975663  4cae31       NaN       10363.20   
4      1781975677  440062  EJU68WB         9601.20   

                                                text  time_delta  
0                                         0EV142235V          11  
1  U45DCI0057Sp-FQ59K1%C6kr0`fl\F'.dNqg'[47,T9!r<iT*          47  
2                J49AJX0031/FUKJJYA.DISB-585028070A7          11  
3                             /EINN.TI2/000EINNA01EA          35  
4                                         0EV142235V           3  


In [13]:
# ==========================================
# SECTION 5: VIEWING THE FUSED SNAPSHOTS (FIXED)
# ==========================================

# 1. Update column selectors to match Pandas auto-generated suffix names
view_columns = [
    'icao24',                  # Unique Aircraft Hex ID
    'callsign',                # Flight Radar Callsign
    'latitude_adsb',           # Physical Latitude Coordinate from Radar
    'longitude_adsb',          # Physical Longitude Coordinate from Radar
    'baro_altitude',           # Physical Altitude (Meters)
    'timestamp_adsb_numeric',  # Radar Snapshot Time (Seconds)
    'timestamp_acars_numeric', # Radio Message Time (Seconds)
    'time_delta',              # Absolute time gap between both signals
    'text'                     # The actual ACARS text message sent by the pilot
]

# 2. Filter using existing columns to guarantee safety
existing_view_cols = [c for c in view_columns if c in final_verified_stream.columns]
clear_view_df = final_verified_stream[existing_view_cols]

# 3. Display the beautiful interactive dataset
clear_view_df.head(15)

,icao24,callsign,latitude_adsb,longitude_adsb,baro_altitude,timestamp_adsb_numeric,timestamp_acars_numeric,time_delta,text
0,440062,EJU68WB,43.0793,9.9544,9738.36,1781975663,1781975674,11,0EV142235V
1,899043,CAL057,13.9101,121.0278,11277.60,1781975663,1781975710,47,"U45DCI0057Sp-FQ59K1%C6kr0`fl\F'.dNqg'[47,T9!r<iT*"
2,8991e5,SJX031,38.4042,142.4319,12192.00,1781975663,1781975674,11,J49AJX0031/FUKJJYA.DISB-585028070A7
3,4cae31,NaN,47.9673,-67.0399,10363.20,1781975663,1781975698,35,/EINN.TI2/000EINNA01EA
4,440062,EJU68WB,43.0567,9.9952,9601.20,1781975677,1781975674,3,0EV142235V
5,899043,CAL057,13.8663,121.0282,11277.60,1781975677,1781975710,33,"U45DCI0057Sp-FQ59K1%C6kr0`fl\F'.dNqg'[47,T9!r<iT*"
6,8991e5,SJX031,38.3786,142.3987,12192.00,1781975677,1781975674,3,J49AJX0031/FUKJJYA.DISB-585028070A7
7,a2d3b3,SWA4330,37.5647,-107.6985,11887.20,1781975677,1781975724,47,"74102,0281,B737-700,260620,WN1049,KBWI,KPHX,20..."
8,4cae31,NaN,47.9673,-67.0399,10363.20,1781975677,1781975698,21,/EINN.TI2/000EINNA01EA
9,440062,EJU68WB,43.0300,10.0480,9425.94,1781975693,1781975674,19,0EV142235V


In [15]:
# ===================================================
# SECTION 6: POST-FUSION EDA & FEATURE SELECTION
# ===================================================

print("--- Post-Fusion Feature Audit ---")
print(f"Starting column count: {final_verified_stream.shape[1]}")

# 1. Inspect the average server sync delay across your dataset
print(f"Average time alignment gap: {final_verified_stream['time_delta'].mean():.2f} seconds\n")

# 2. Select only the core features needed for your AI Agent
core_features = [
    'timestamp_adsb_numeric', # Synced Radar Unix time
    'icao24',                  # Unique 24-bit aircraft hex address
    'callsign',                # Flight callsign from radar
    'origin_country',          # Ground-truth origin country
    'latitude_adsb',           # Spatial Y coordinate
    'longitude_adsb',          # Spatial X coordinate
    'baro_altitude',           # Ground-truth elevation in meters
    'velocity',                # Aircraft ground speed in m/s
    'tail',                    # Reg tail number from radio channel
    'flightNumber',            # Flight number from radio channel
    'frequency',               # Radio link frequency channel
    'text'                     # The actual text payload sent by the pilot
]

# 3. Create your final, lean, clean master machine learning dataset
clean_master_df = final_verified_stream[core_features].copy()

# 4. Handle any remaining missing values in essential tracking fields
clean_master_df['callsign'] = clean_master_df['callsign'].fillna('UNKNOWN')
clean_master_df = clean_master_df.dropna(subset=['latitude_adsb', 'longitude_adsb', 'baro_altitude'])

print("--- Final Production Dataset Profile ---")
print(f"Final column count: {clean_master_df.shape[1]}")
print(f"Final valid rows ready for AI engine: {clean_master_df.shape[0]}")
clean_master_df.head(5)

--- Post-Fusion Feature Audit ---
Starting column count: 44
Average time alignment gap: 29.85 seconds

--- Final Production Dataset Profile ---
Final column count: 12
Final valid rows ready for AI engine: 1329


,timestamp_adsb_numeric,icao24,callsign,origin_country,latitude_adsb,longitude_adsb,baro_altitude,velocity,tail,flightNumber,frequency,text
0,1781975663,440062,EJU68WB,Austria,43.0793,9.9544,9738.36,223.17,OE-IVF,EC85JE,136.825,0EV142235V
1,1781975663,899043,CAL057,Taiwan,13.9101,121.0278,11277.60,260.84,B-18920,CI57,NaN,"U45DCI0057Sp-FQ59K1%C6kr0`fl\F'.dNqg'[47,T9!r<iT*"
2,1781975663,8991e5,SJX031,Taiwan,38.4042,142.4319,12192.00,221.91,B-58502,JX31,NaN,J49AJX0031/FUKJJYA.DISB-585028070A7
3,1781975663,4cae31,UNKNOWN,Ireland,47.9673,-67.0399,10363.20,229.34,EI-XLT,EI09KN,136.975,/EINN.TI2/000EINNA01EA
4,1781975677,440062,EJU68WB,Austria,43.0567,9.9952,9601.20,226.03,OE-IVF,EC85JE,136.825,0EV142235V


In [17]:
# ==========================================
# SECTION 6: EXPORT AND DOWNLOAD FULL DATASET
# ==========================================

# 1. Save the full fused dataset to a clean CSV layout inside Colab's virtual drive
final_verified_stream.to_csv("fused_aircraft_tracking_data.csv", index=False)

# 2. Trigger an automatic browser download prompt on your Windows machine
from google.colab import files
files.download("fused_aircraft_tracking_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>